# Rad s modelima Meta obitelji 

## Uvod 

Ova lekcija će obuhvatiti: 

- Istraživanje dva glavna Meta obitelj modela - Llama 3.1 i Llama 3.2, plus trenutno dostupnih alternativa korištenih u primjerima koda (Llama 3.3 i Llama 4) 
- Razumijevanje slučajeva uporabe i scenarija za svaki model 
- Primjer koda koji pokazuje jedinstvene značajke svakog modela 


## Meta obitelj modela 

U ovoj lekciji, istražit ćemo 2 modela iz Meta obitelji ili "Llama stada" - Llama 3.1 i Llama 3.2. Izvedivi primjeri koda koriste Llama 3.3 i Llama 4, koji su trenutno dostupne alternative u Microsoft Foundry Models katalogu. 

Ovi modeli dolaze u različitim varijantama i dostupni su u [Microsoft Foundry Models katalogu](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Napomena:** GitHub Models se ukida krajem srpnja 2026. Evo više detalja o korištenju [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) za prototipiranje s AI modelima.

> **Napomena:** Specifične verzije modela za koje je ova lekcija originalno napisana (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) otada su povučene iz Microsoft Foundry Models kataloga. Primjeri koda u nastavku koriste `Llama-3.3-70B-Instruct` i `Llama-4-Scout-17B-16E-Instruct` kao trenutno dostupne alternative u Microsoft Foundry Models katalogu, pa i danas rade.

Varijante modela (kako je prvotno opisano u ovoj lekciji): 
- Llama 3.1 - 70B Instruct (povučen) 
- Llama 3.1 - 405B Instruct (povučen) 
- Llama 3.2 - 11B Vision Instruct (povučen) 
- Llama 3.2 - 90B Vision Instruct (povučen) 

Trenutno dostupne alternative korištene u primjerima koda: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Napomena: Varijante Llama 3 gore navedene zadržane su radi povijesnog konteksta i više nisu dostupne u Microsoft Foundry Models.*



## Llama 3.1 

Sa 405 milijardi parametara, Llama 3.1 spada u kategoriju otvorenih LLM modela. 

Ovaj model je nadogradnja ranije verzije Llama 3, nudeći: 

- Veće kontekstno okno - 128k tokena naspram 8k tokena 
- Veći maksimalni broj izlaznih tokena - 4096 naspram 2048 
- Bolju podršku za više jezika - zbog povećanja broja tokena u treningu 

To omogućuje Llama 3.1 da se nosi sa složenijim slučajevima upotrebe pri izgradnji GenAI aplikacija, uključujući: 
- Izvorno pozivanje funkcija - mogućnost pozivanja vanjskih alata i funkcija izvan LLM toka rada
- Bolje performanse RAG-a - zbog većeg kontekstnog okna 
- Generiranje sintetičkih podataka - mogućnost stvaranja učinkovitih podataka za zadatke poput finog podešavanja 



### Pozivanje izvornih funkcija 

Llama 3.1 je dodatno podešen da bude učinkovitiji u pozivanju funkcija ili alata. Također ima dva ugrađena alata koja model može prepoznati kao potrebna za korištenje na temelju upita korisnika. Ti alati su: 

- **Brave Search** - Može se koristiti za dobivanje ažurnih informacija poput vremenske prognoze obavljanjem pretraživanja na internetu 
- **Wolfram Alpha** - Može se koristiti za složenije matematičke izračune tako da nije potrebno pisati vlastite funkcije. 

Također možete kreirati vlastite prilagođene alate koje LLM može pozivati. 

U donjem primjeru koda: 

- Definiramo dostupne alate (brave_search, wolfram_alpha) u sistemskoj prompti. 
- Šaljemo korisnički upit koji pita o vremenu u određenom gradu. 
- LLM će odgovoriti pozivom alata Brave Search koji će izgledati ovako `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Napomena: Ovaj primjer samo obavlja poziv alata, ako želite dobiti rezultate, potrebno je kreirati besplatan račun na Brave API stranici i definirati samu funkciju` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Iako je LLM, jedno ograničenje koje Llama 3.1 ima je multimodalnost. To znači mogućnost korištenja različitih vrsta ulaza poput slika kao upita i pružanja odgovora. Ova sposobnost jedna je od glavnih značajki Llama 3.2. Te značajke također uključuju: 

- Multimodalnost - ima sposobnost evaluacije i tekstualnih i slikovnih upita 
- Varijacije malih do srednjih veličina (11B i 90B) - to pruža fleksibilne opcije implementacije, 
- Samo tekstualne varijacije (1B i 3B) - to omogućuje implementaciju modela na edge / mobilnim uređajima i pruža malu latenciju 

Podrška za multimodalnost predstavlja veliki korak u svijetu open source modela. Donji primjer koda koristi i sliku i tekstualni upit za dobivanje analize slike. Budući da je Llama 3.2 90B Vision Instruct povučena, primjer koristi `Llama-4-Scout-17B-16E-Instruct`, trenutno dostupnu nativno multimodalnu alternativu. 

### Multimodalna podrška s Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Učenje ovdje ne prestaje, nastavi putovanje

Nakon što dovršiš ovu lekciju, pogledaj našu [kolekciju za učenje Generativne AI](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) kako bi nastavio unapređivati svoje znanje o Generativnoj AI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
